# Event Dummy

**4가지 데이터셋 생성 스크립트**
- EDA_1st_result.csv : 원본 1st data (레벨값)
- Derived_Variable.csv : 1st data가 1차 차분된 값 + 파생변수들 (oil_diff_target 포함)

**생성되는 데이터셋** (모두 타겟 변수는 oil_diff_target = OilPrice.diff().shift(-1)):
1) dataset1_raw_only        : 1st data 레벨값 + oil_diff_target (OilPrice 제외)
2) dataset2_raw_plus_derived: 1st data 레벨값 + 파생변수 + oil_diff_target (OilPrice 제외)
3) dataset3_diff_only       : 1st data 1차 차분 + oil_diff_target (차분된 OilPrice 제외)
4) dataset4_derived_full    : Derived_Variable 그대로 (차분된 1st + 파생변수 + target)

In [ ]:
import pandas as pd
from pathlib import Path

# ===== 경로 설정 =====
INPUT_DIR = Path("../data/Finance_Final")
OUTPUT_DIR = Path("../data/Finance_Final")  # 사용자 지정 상대경로
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EDA_PATH = INPUT_DIR / "EDA_1st_result.csv"
DERIVED_PATH = INPUT_DIR / "Derived_Variable.csv"

TARGET_COL = "oil_diff_target"  # OilPrice 대신 사용할 타겟 변수


# ===== 데이터 로드 =====
df_raw = pd.read_csv(EDA_PATH, parse_dates=["date"])
df_derived = pd.read_csv(DERIVED_PATH, parse_dates=["date"])

print(f"[원본 1st data]      shape={df_raw.shape}, "
      f"기간={df_raw['date'].min().date()} ~ {df_raw['date'].max().date()}")
print(f"[Derived_Variable]   shape={df_derived.shape}, "
      f"기간={df_derived['date'].min().date()} ~ {df_derived['date'].max().date()}")


# ===== 컬럼 분류 =====
raw_cols = [c for c in df_raw.columns if c != "date"]

# 파생변수 컬럼 (Derived에만 있는 것)
derived_only_cols = [c for c in df_derived.columns if c not in df_raw.columns]

# Derived 안에서 1st data와 동일한 이름의 컬럼 (이미 1차 차분된 값)
diff_cols_in_derived = [c for c in df_raw.columns
                        if c in df_derived.columns and c != "date"]


def reorder_target_first(df: pd.DataFrame) -> pd.DataFrame:
    """date → oil_diff_target → 나머지 순으로 컬럼 정렬"""
    others = [c for c in df.columns if c not in ("date", TARGET_COL)]
    return df[["date", TARGET_COL] + others]


# ===== 1) 1st data만 (OilPrice → oil_diff_target 으로 교체) =====
dataset1 = df_raw.merge(
    df_derived[["date", TARGET_COL]],
    on="date",
    how="inner",
)
dataset1 = dataset1.drop(columns=["OilPrice"])
dataset1 = reorder_target_first(dataset1)


# ===== 2) 1st data + 파생변수 (OilPrice → oil_diff_target 으로 교체) =====
# 파생변수에 이미 oil_diff_target 포함되어 있음
dataset2 = df_raw.merge(
    df_derived[["date"] + derived_only_cols],
    on="date",
    how="inner",
)
dataset2 = dataset2.drop(columns=["OilPrice"])
dataset2 = reorder_target_first(dataset2)


# ===== 3) 1st data의 1차 차분만 (차분된 OilPrice → oil_diff_target 으로 교체) =====
# Derived에서 1st data와 동명 컬럼들(차분값) + oil_diff_target 만 사용
dataset3 = df_derived[["date"] + diff_cols_in_derived + [TARGET_COL]].copy()
dataset3 = dataset3.drop(columns=["OilPrice"])  # 차분된 OilPrice도 제거
dataset3 = reorder_target_first(dataset3)


# ===== 4) Derived_Variable 그대로 =====
dataset4 = df_derived.copy()
dataset4 = reorder_target_first(dataset4)


# ===== 저장 =====
out_files = {
    "dataset1_raw_only.csv":         dataset1,
    "dataset2_raw_plus_derived.csv": dataset2,
    "dataset3_diff_only.csv":        dataset3,
    "dataset4_derived_full.csv":     dataset4,
}

print(f"\n===== 저장 경로: {OUTPUT_DIR.resolve()} =====")
for fname, df in out_files.items():
    fpath = OUTPUT_DIR / fname
    df.to_csv(fpath, index=False)
    print(f"{fname:35s}  shape={df.shape}")


# ===== 요약 =====
print("\n===== 데이터셋 설명 (모두 타겟: oil_diff_target) =====")
print("1) dataset1_raw_only         : 1st data 레벨값 (OilPrice 제외) + oil_diff_target")
print("2) dataset2_raw_plus_derived : 1st data 레벨값 (OilPrice 제외) + 파생변수 (target 포함)")
print("3) dataset3_diff_only        : 1st data 1차 차분 (차분된 OilPrice 제외) + oil_diff_target")
print("4) dataset4_derived_full     : 1st data 1차 차분 + 파생변수 (target 포함)")

# 컬럼 확인용 출력
print(f"\n[dataset1 컬럼] {list(dataset1.columns)}")
print(f"[dataset2 컬럼] {list(dataset2.columns)}")
print(f"[dataset3 컬럼] {list(dataset3.columns)}")
print(f"[dataset4 컬럼] {list(dataset4.columns)}")

[원본 1st data]      shape=(4820, 14), 기간=2007-01-02 ~ 2026-03-16
[Derived_Variable]   shape=(4799, 28), 기간=2007-02-01 ~ 2026-03-16

===== 저장 경로: C:\Users\SAMSUNG\OneDrive\바탕 화면\비어플\Final_file\BAF-Finance2\data\Finance_Final =====
dataset1_raw_only.csv                shape=(4799, 14)
dataset2_raw_plus_derived.csv        shape=(4799, 28)
dataset3_diff_only.csv               shape=(4799, 13)
dataset4_derived_full.csv            shape=(4799, 28)

===== 데이터셋 설명 (모두 타겟: oil_diff_target) =====
1) dataset1_raw_only         : 1st data 레벨값 (OilPrice 제외) + oil_diff_target
2) dataset2_raw_plus_derived : 1st data 레벨값 (OilPrice 제외) + 파생변수 (target 포함)
3) dataset3_diff_only        : 1st data 1차 차분 (차분된 OilPrice 제외) + oil_diff_target
4) dataset4_derived_full     : 1st data 1차 차분 + 파생변수 (target 포함)

[dataset1 컬럼] ['date', 'oil_diff_target', 'RealInterestRate', 'CPI', 'DollarIndex', 'VIX', 'IndustryProduction', 'CPE', 'OilInventories', 'OPECProduction', 'OilProduction', 'TermSpread', 'TreasuryYield', 'Fed

## ** 공통 부분 - 하드코딩

In [4]:
## 하드코딩
#방식 1. 단기충격 + 장기국면
#방식 2. 단기충격 중심
#방식 3. 이벤트 윈도우
import pandas as pd
import numpy as np

# --------------------------------------------
# 0. 데이터 준비
# --------------------------------------------
dataset_map = {
    'dataset1_raw_only': dataset1,
    'dataset2_raw_plus_derived': dataset2,
    'dataset3_diff_only': dataset3,
    'dataset4_derived_full': dataset4,
}

def prepare_dataset(df):
    df_out = df.copy()

    if not isinstance(df_out.index, pd.DatetimeIndex):
        if 'date' in df_out.columns:
            df_out['date'] = pd.to_datetime(df_out['date'])
            df_out = df_out.set_index('date').sort_index()
        else:
            raise ValueError("DatetimeIndex 또는 date 컬럼이 필요합니다.")

    return df_out

# 4가지 데이터셋 모두에 동일한 전처리 적용
prepared_datasets = {
    name: prepare_dataset(df)
    for name, df in dataset_map.items()
}

# 확인
for name, df in prepared_datasets.items():
    print(
        f"{name:30s} | shape={df.shape} | "
        f"period={df.index.min().date()} ~ {df.index.max().date()}"
    )

dataset1_raw_only              | shape=(4799, 13) | period=2007-02-01 ~ 2026-03-16
dataset2_raw_plus_derived      | shape=(4799, 27) | period=2007-02-01 ~ 2026-03-16
dataset3_diff_only             | shape=(4799, 12) | period=2007-02-01 ~ 2026-03-16
dataset4_derived_full          | shape=(4799, 27) | period=2007-02-01 ~ 2026-03-16


In [5]:
# --------------------------------------------
# 1. 이벤트 기준일 / 기간 정의
# --------------------------------------------
# 기준일(anchor): 사건 발생 혹은 시장 충격이 본격화된 시점
# shock: 단기 충격 구간 (급격한 시장 반응)
# regime: 이후 지속되는 구조적 변화 구간,장기국면 (shock 이후 구간만 따로
#→ “충격(Shock) + 이후 체제 변화(Regime)를 나눠서 분석하겠다”는 설계

event_periods = {
    'gfc_2008': {
        'anchor': '2008-09-15',   # 리먼 사태 <기준일>
        'shock_start': '2008-09-15', #<단기 충격 구간>
        'shock_end':   '2008-12-31',
        'regime_start':'2009-01-01', #<장기 국면 구간>
        'regime_end':  '2009-06-30'
    },
    'opec_2014': { 
        'anchor': '2014-11-27',   # OPEC 감산 대신 점유율 유지 
        'shock_start': '2014-11-27',
        'shock_end':   '2015-01-31',
        'regime_start':'2015-02-01',
        'regime_end':  '2016-02-29'
    },
    'covid_2020': {
        'anchor': '2020-03-11',   # WHO 팬데믹 선언
        'shock_start': '2020-03-11',
        'shock_end':   '2020-05-31',
        'regime_start':'2020-06-01',
        'regime_end':  '2020-12-31'
    },
    'war_2022': {
        'anchor': '2022-02-24',   # 러시아-우크라이나 전쟁 발발
        'shock_start': '2022-02-24',
        'shock_end':   '2022-04-30',
        'regime_start':'2022-05-01',
        'regime_end':  '2022-12-31'
    }
}

# --------------------------------------------
# 2. 공통 함수
#본 분석에서는 이벤트 발생 이전 구간을 더미에 포함하지 않았다. 
#사건 발생 이전 기간까지 포함할 경우, 모델이 사건 발생을 사전에 알고 있었다는 해석상의 문제가 발생할 수 있기 때문 
#따라서 이벤트 윈도우는 사건 발생일을 기준으로 이후 20영업일로 설정하여, 사건 이후 시장에 반영되는 단기 충격을 포착하고자 하였다.
# --------------------------------------------
def add_period_dummy(df_in, col_name, start, end):
    df_in[col_name] = ((df_in.index >= pd.to_datetime(start)) &
                        (df_in.index <= pd.to_datetime(end))).astype(int)
    return df_in

def add_window_dummy(df_in, col_name, anchor, window_bdays=20):
    anchor = pd.to_datetime(anchor)
    start = anchor
    end   = anchor + pd.offsets.BDay(window_bdays)
    df_in[col_name] = ((df_in.index >= start) & (df_in.index <= end)).astype(int)
    return df_in
#df_in 은 anchor <= 날짜 <= anchor + 20영업일

# --------------------------------------------
# 3. 방식 1,2,3에 따른 변수 생성 함수
# --------------------------------------------

def make_hard_dummies(index):
    
    # 방식 1: 단기충격 + 장기국면
    df_hard_sl = pd.DataFrame(index=index)

    for evt, info in event_periods.items():
        # 단기충격
        df_hard_sl = add_period_dummy(
            df_hard_sl,
            f'{evt}_shock',
            info['shock_start'],
            info['shock_end']
        )
        
        # 장기국면 (shock 이후만 따로)
        df_hard_sl = add_period_dummy(
            df_hard_sl,
            f'{evt}_regime',
            info['regime_start'],
            info['regime_end']
        )

    hard_sl_cols = [c for c in df_hard_sl.columns if c.endswith('_shock') or c.endswith('_regime')]

    # 방식 2: 단기충격 중심
    df_hard_short = pd.DataFrame(index=index)

    for evt, info in event_periods.items():
        df_hard_short = add_period_dummy(
            df_hard_short,
            f'{evt}_short',
            info['shock_start'],
            info['shock_end']
        )

    hard_short_cols = [c for c in df_hard_short.columns if c.endswith('_short')]

    # 방식 3: 이벤트 윈도우 (+20 영업일)
    df_hard_window = pd.DataFrame(index=index)

    for evt, info in event_periods.items():
        df_hard_window = add_window_dummy(
            df_hard_window,
            f'{evt}_window',
            info['anchor'],
            window_bdays=20
        )

    hard_window_cols = [c for c in df_hard_window.columns if c.endswith('_window')]

    return {
        'shock_plus_regime': {
            'df': df_hard_sl,
            'cols': hard_sl_cols
        },
        'short_only': {
            'df': df_hard_short,
            'cols': hard_short_cols
        },
        'event_window': {
            'df': df_hard_window,
            'cols': hard_window_cols
        }
    }

In [6]:
hard_sets_by_dataset = {
    name: make_hard_dummies(df.index)
    for name, df in prepared_datasets.items()
}

for dataset_name, hard_sets in hard_sets_by_dataset.items():
    print(f"\n===== {dataset_name} =====")

    for scheme_name, info in hard_sets.items():
        print(f"\n[{scheme_name}]")
        print(info['cols'])
        print(info['df'][info['cols']].sum().sort_index())


===== dataset1_raw_only =====

[shock_plus_regime]
['gfc_2008_shock', 'gfc_2008_regime', 'opec_2014_shock', 'opec_2014_regime', 'covid_2020_shock', 'covid_2020_regime', 'war_2022_shock', 'war_2022_regime']
covid_2020_regime    149
covid_2020_shock      56
gfc_2008_regime      124
gfc_2008_shock        76
opec_2014_regime     271
opec_2014_shock       43
war_2022_regime      169
war_2022_shock        46
dtype: int64

[short_only]
['gfc_2008_short', 'opec_2014_short', 'covid_2020_short', 'war_2022_short']
covid_2020_short    56
gfc_2008_short      76
opec_2014_short     43
war_2022_short      46
dtype: int64

[event_window]
['gfc_2008_window', 'opec_2014_window', 'covid_2020_window', 'war_2022_window']
covid_2020_window    21
gfc_2008_window      21
opec_2014_window     19
war_2022_window      21
dtype: int64

===== dataset2_raw_plus_derived =====

[shock_plus_regime]
['gfc_2008_shock', 'gfc_2008_regime', 'opec_2014_shock', 'opec_2014_regime', 'covid_2020_shock', 'covid_2020_regime', 'w

In [7]:
# --------------------------------------------
# 하드코딩 이벤트 더미 생성 결과 확인
# --------------------------------------------

for dataset_name, hard_sets in hard_sets_by_dataset.items():
    print(f"\n==============================")
    print(f"{dataset_name}")
    print(f"==============================")

    # 이벤트 윈도우
    df_hard_window = hard_sets['event_window']['df']
    hard_window_cols = hard_sets['event_window']['cols']

    print("\n[이벤트 윈도우 변수]")
    print(hard_window_cols)
    print(df_hard_window[hard_window_cols].sum().sort_index())
    # hard_window_cols 는 anchor 날짜부터 +20 영업일(BDay)

    # 단기충격 + 장기국면
    df_hard_sl = hard_sets['shock_plus_regime']['df']
    hard_sl_cols = hard_sets['shock_plus_regime']['cols']

    print("\n[단기충격+장기국면 변수]")
    print(hard_sl_cols)
    print(df_hard_sl[hard_sl_cols].sum().sort_index())
    # hard_sl_cols 는 단기+장기

    # 단기충격 중심
    df_hard_short = hard_sets['short_only']['df']
    hard_short_cols = hard_sets['short_only']['cols']

    print("\n[단기충격 중심 변수]")
    print(hard_short_cols)
    print(df_hard_short[hard_short_cols].sum().sort_index())
    # hard_short_cols 는 단기



dataset1_raw_only

[이벤트 윈도우 변수]
['gfc_2008_window', 'opec_2014_window', 'covid_2020_window', 'war_2022_window']
covid_2020_window    21
gfc_2008_window      21
opec_2014_window     19
war_2022_window      21
dtype: int64

[단기충격+장기국면 변수]
['gfc_2008_shock', 'gfc_2008_regime', 'opec_2014_shock', 'opec_2014_regime', 'covid_2020_shock', 'covid_2020_regime', 'war_2022_shock', 'war_2022_regime']
covid_2020_regime    149
covid_2020_shock      56
gfc_2008_regime      124
gfc_2008_shock        76
opec_2014_regime     271
opec_2014_shock       43
war_2022_regime      169
war_2022_shock        46
dtype: int64

[단기충격 중심 변수]
['gfc_2008_short', 'opec_2014_short', 'covid_2020_short', 'war_2022_short']
covid_2020_short    56
gfc_2008_short      76
opec_2014_short     43
war_2022_short      46
dtype: int64

dataset2_raw_plus_derived

[이벤트 윈도우 변수]
['gfc_2008_window', 'opec_2014_window', 'covid_2020_window', 'war_2022_window']
covid_2020_window    21
gfc_2008_window      21
opec_2014_window     19
war_20

In [ ]:
# 이벤트 이름 추출
events = [c.replace('_shock', '') for c in hard_sl_cols if c.endswith('_shock')]

summary = pd.DataFrame({
    'shock': [
        df_hard_sl[f'{evt}_shock'].sum()
        for evt in events
    ],
    'regime': [
        df_hard_sl[f'{evt}_regime'].sum()
        for evt in events
    ],
    'short_only': [
        df_hard_short[f'{evt}_short'].sum()
        for evt in events
    ]
}, index=events)

print(summary)

In [ ]:
print("\n[단기충격+장기국면]")
print(df_hard_sl[hard_sl_cols].sum())

print("\n[단기충격 중심]")
print(df_hard_short[hard_short_cols].sum())

print("\n[이벤트 윈도우]")
print(df_hard_window[hard_window_cols].sum())

## 1) dataset1_raw_only

## 2) dataset2_raw_plus_derived

## 3) dataset3_diff_only

## 4) dataset4_derived_full